# Calculate ETCCDI climate indices — MIROC-ES2H

This code calculates the climate indices for MIROC-ES2H experiments (G6-1.5K-SAI and baseline),
reading data from the S3 bucket `s3://reflective-persistent-prod-large/MIROC-ES2H/G6-1.5K-SAI/day/`.

You will need:
- daily maximum temperature (`tasmax`)
- daily minimum temperature (`tasmin`)
- daily precipitation rate (`pr`)

This code generates the following output (see tables below) in the specified folder.
The indices calculated here do not include percentile-based indices.

SOURCE: https://xclim.readthedocs.io/en/stable/indices.html#xclim.indices.prcptot_wetdry_period

Notebook is not fast. It takes ~ 5 mins to do the temperature indices, for 1 ensemble member, so ~ 45 mins for all temperature indices, and 90 mins for all. 

In [1]:
import xarray as xr
import numpy as np
from os.path import join, expanduser
import xclim.indices
import os
import gc
import s3fs
import fsspec
from pathlib import Path
import cftime
import dask

In [2]:
# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

model='MIROC-ES2H'

S3_BASE = f's3://reflective-persistent-prod-large/{model}/'  # scenario appended per-loop

SCENARIOS = ['G6-1.5K-SAI', 'G6-1.5K-HiLLA', 'baseline']
ENSEMBLE_MEMBERS = ['r01', 'r02', 'r03']

# Root output directory — adjust as needed
OUTPUT_ROOT = f'./out_data/ETCCDI_indices_annual/{model}/'

fs = s3fs.S3FileSystem(anon=True)

# Chunk along time by one year (365 days).
# MIROC-ES2H T85 grid: 128 lat × 256 lon; 365 × 128 × 256 × 4 B ≈ 48 MB per chunk,
# which is efficient for S3 reads and aligns exactly with freq='YS' resampling.
CHUNKS = {'time': 365}

## First calculate fixed threshold and index indices

### Temperature indices

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| tasmax (Tx) | SU | Summer days | Number of days when Tx $>$ 25$^\circ$C | days per year | Fixed threshold |
| tasmax (Tx) | ID | Ice days | Number of days when Tx $<$ 0$^\circ$C | days per year | Fixed threshold |
| tasmax (Tx) | TXn | Coldest days | Annual minimum daily maximum temperature | $^\circ$C | Fixed Index |
| tasmax (Tx) | TXx | Warmest days | Annual maximum daily maximum temperature | $^\circ$C | Fixed Index |
| tasmin (Tn) | TNn | Coldest night | Annual minimum daily minimum temperature | $^\circ$C | Fixed Index |
| tasmin (Tn) | TNx | Warmest night | Annual maximum daily minimum temperature | days per year | Fixed Index |
| tasmin (Tn) | FD | Frost days | Number of days when Tn < 0$^\circ$C | days per year | Fixed threshold |
| tasmin (Tn) | TN | Tropical nights | Number of days when Tn $>$ 20$^\circ$C | days per year | Fixed threshold |

In [33]:
for scenario in SCENARIOS:
#for scenario in ['baseline']:
    for member in ENSEMBLE_MEMBERS:
        print(f'Processing scenario={scenario}, member={member}')

        if scenario in ['G6-1.5K-SAI', 'G6-1.5K-HiLLA']:
            s3_base = f'{S3_BASE}{scenario}/day/'
        elif scenario == 'baseline':
            s3_base = f'{S3_BASE}G6-1.5K-SAI/day/'
            
        tasmax_path = f'{s3_base}tasmax_{scenario}_{member}.nc'
        tasmin_path = f'{s3_base}tasmin_{scenario}_{member}.nc'

        tasmax_ds = xr.open_dataset(
            fsspec.open(tasmax_path, mode='rb', anon=False).open(),
            engine='h5netcdf', chunks=CHUNKS
        )
        tasmin_ds = xr.open_dataset(
            fsspec.open(tasmin_path, mode='rb', anon=False).open(),
            engine='h5netcdf', chunks=CHUNKS
        )

        # Remove duplicate time steps if any
        tasmax_ds = tasmax_ds.sel(time=~tasmax_ds.get_index('time').duplicated())
        tasmin_ds = tasmin_ds.sel(time=~tasmin_ds.get_index('time').duplicated())

        tx = tasmax_ds['tasmax']  # units: K (CMIP6 convention)
        tn = tasmin_ds['tasmin']  # units: K

        # =======================================================
        # Yearly indices
        # =======================================================

        # Number of days when Tx >= 25°C (SU)
        SU = xclim.indices.tx_days_above(tx, thresh='25.0 degC', freq='YS')

        # Number of ice/freezing days (ID)
        ID = xclim.indices.ice_days(tx, thresh='0 degC', freq='YS')

        # Annual Maximum Daily Maximum / Hottest Day (TXX)
        TXX = xclim.indices.tx_max(tx, freq='YS')

        # Annual Minimum Daily Maximum / Coldest Day (TXN)
        TXN = xclim.indices.tx_min(tx, freq='YS')

        # Annual count of frost days (TN < 0°C)
        FD = xclim.indices.frost_days(tn, freq='YS')

        # Annual count of tropical nights (TN > 20°C)
        TN = xclim.indices.tn_days_above(tn, thresh='20.0 degC', freq='YS')

        # Annual maximum of daily minimum temperature (TNX: warmest night)
        TNX = xclim.indices.tn_max(tn, freq='YS')

        # Annual minimum of daily minimum temperature (TNN: coldest night)
        TNN = xclim.indices.tn_min(tn, freq='YS')

        su, id_, txx, txn, fd, tn_, tnx, tnn = dask.compute(SU, ID, TXX, TXN, FD, TN, TNX, TNN)

        # Output directory for this scenario/member
        out_dir = join(OUTPUT_ROOT, scenario, member)
        os.makedirs(out_dir, exist_ok=True)

        su.to_netcdf(join(out_dir, 'SU.nc'))
        id_.to_netcdf(join(out_dir, 'ID.nc'))
        txx.to_netcdf(join(out_dir, 'TXX.nc'))
        txn.to_netcdf(join(out_dir, 'TXN.nc'))
        fd.to_netcdf(join(out_dir, 'FD.nc'))
        tn_.to_netcdf(join(out_dir, 'TN.nc'))
        tnx.to_netcdf(join(out_dir, 'TNX.nc'))
        tnn.to_netcdf(join(out_dir, 'TNN.nc'))

        print(f'  -> Temperature indices saved to {out_dir}')

Processing scenario=baseline, member=r01
  -> Temperature indices saved to ./out_data/ETCCDI_indices_annual/MIROC-ES2H/baseline/r01
Processing scenario=baseline, member=r02
  -> Temperature indices saved to ./out_data/ETCCDI_indices_annual/MIROC-ES2H/baseline/r02
Processing scenario=baseline, member=r03
  -> Temperature indices saved to ./out_data/ETCCDI_indices_annual/MIROC-ES2H/baseline/r03


### Precipitation indices

In [3]:
# Free memory from the temperature loop before starting precipitation
try:
    del tasmax_ds, tasmin_ds, tx, tn
    del su, id_, txx, txn, fd, tn_, tnx, tnn
except NameError:
    pass
gc.collect()

499

In [4]:
# To convert precipitation rate to mm/day
SECONDS_PER_DAY = 86400
M_TO_MM = 1000

def to_mm_day(da, varname=None):
    """
    Convert precipitation rate to mm/day when units look like:
      - kg m-2 s-1  (water flux; 1 kg/m^2 = 1 mm water)
      - m s-1       (water-equivalent depth rate)
    Otherwise return da unchanged.
    """
    units = (da.attrs.get("units") or "").strip().lower().replace(" ", "")

    is_kg_m2_s = units in {"kgm-2s-1", "kg/m^2/s", "kg/m2/s", "kgm^-2s^-1", "kgm**-2s**-1"}
    is_m_s     = units in {"ms-1", "m/s", "m s-1", "m/s-1"} or units == "ms^-1"

    if is_kg_m2_s:
        out = da * SECONDS_PER_DAY  # kg/m^2/day == mm/day
    elif is_m_s:
        out = da * SECONDS_PER_DAY * M_TO_MM
    else:
        if varname is not None and varname.upper() in {"PRECT", "PRECC", "PRECL", "PR"}:
            out = da * SECONDS_PER_DAY
        else:
            return da

    out = out.assign_attrs(da.attrs)
    out.attrs["units"] = "mm/day"
    return out

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| pr (PR) | PRCPTOT | Total rainfall | Annual sum of precipitation | mm | Fixed Index |
| pr (PR) | SDII | Simple daily intensity | Mean precipitation falling on days when PR $\geq$ 1 mm | mm | Fixed Index |
| pr (PR) | Rx1day | Wettest day | Annual maximum precipitation in a single day | mm | Fixed Index |
| pr (PR) | Rx5day | Wettest pentad | Annual maximum precipitation falling on 5 consecutive days | mm | Fixed Index |
| pr (PR) | CDD | Consecutive dry days | Longest spell of consecutive days when PR $\leq$ 1 mm | days per year | Fixed index/spell |
| pr (PR) | CWD | Consecutive wet days | Longest spell of consecutive days when PR $\geq$ 1 mm | days per year | Fixed index/spell |
| pr (PR) | R10mm | Heavy precipitation days | Number of days when precipitation $\geq$ 10 mm | days per year | Fixed threshold |
| pr (PR) | R20mm | Very heavy precipitation days | Number of days when precipitation $\geq$ 20 mm | days per year | Fixed threshold |

In [5]:
import warnings

for scenario in SCENARIOS:
    for member in ENSEMBLE_MEMBERS:
        print(f'Processing scenario={scenario}, member={member}')

        if scenario in ['G6-1.5K-SAI', 'G6-1.5K-HiLLA']:
            s3_base = f'{S3_BASE}{scenario}/day/'
        elif scenario == 'baseline':
            s3_base = f'{S3_BASE}G6-1.5K-SAI/day/'

        pr_path = f'{s3_base}pr_{scenario}_{member}.nc'

        ds = xr.open_dataset(
            fsspec.open(pr_path, mode='rb', anon=False).open(),
            engine='h5netcdf', chunks=CHUNKS
        )

        ds = ds.sel(time=~ds.get_index('time').duplicated())
        pr = to_mm_day(ds['pr'], varname='pr')  # convert to mm/day

        # =======================================================
        # Yearly indices — batch 1: simple aggregations
        # =======================================================

        PRCPTOT = xclim.indices.prcptot(pr, freq='YS')
        RX1D    = xclim.indices.max_1day_precipitation_amount(pr, freq='YS')
        RX5D    = xclim.indices.max_n_day_precipitation_amount(pr, window=5, freq='YS')
        R10MM   = xclim.indices.wetdays(pr, thresh='10 mm/day', freq='YS', op='>=')
        R20MM   = xclim.indices.wetdays(pr, thresh='20 mm/day', freq='YS', op='>=')

        # SDII divides by wet-day count; NaN where no wet days occur is expected
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            SDII = xclim.indices.daily_pr_intensity(pr, thresh='1 mm/day', freq='YS')

        prcptot, sdii, rx1d, rx5d, r10mm, r20mm = dask.compute(
            PRCPTOT, SDII, RX1D, RX5D, R10MM, R20MM
        )

        # =======================================================
        # Yearly indices — batch 2: spell-based (rolling window, higher memory)
        # =======================================================

        CDD = xclim.indices.maximum_consecutive_dry_days(pr, '1 mm/day', freq='YS')
        cdd, = dask.compute(CDD)

        CWD = xclim.indices.maximum_consecutive_wet_days(pr, thresh='1 mm/day', freq='YS')
        cwd, = dask.compute(CWD)

        # =======================================================
        # Save
        # =======================================================

        out_dir = join(OUTPUT_ROOT, scenario, member)
        os.makedirs(out_dir, exist_ok=True)

        prcptot.to_netcdf(join(out_dir, 'PRCPTOT.nc'))
        sdii.to_netcdf(join(out_dir, 'SDII.nc'))
        rx1d.to_netcdf(join(out_dir, 'RX1D.nc'))
        rx5d.to_netcdf(join(out_dir, 'RX5D.nc'))
        cdd.to_netcdf(join(out_dir, 'CDD.nc'))
        cwd.to_netcdf(join(out_dir, 'CWD.nc'))
        r10mm.to_netcdf(join(out_dir, 'R10MM.nc'))
        r20mm.to_netcdf(join(out_dir, 'R20MM.nc'))

        del ds, pr, prcptot, sdii, rx1d, rx5d, r10mm, r20mm, cdd, cwd
        gc.collect()

        print(f'  -> Precipitation indices saved to {out_dir}')

Processing scenario=G6-1.5K-SAI, member=r01


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/MIROC-ES2H/G6-1.5K-SAI/r01
Processing scenario=G6-1.5K-SAI, member=r02


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/MIROC-ES2H/G6-1.5K-SAI/r02
Processing scenario=G6-1.5K-SAI, member=r03


...

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/MIROC-ES2H/baseline/r01
Processing scenario=baseline, member=r02


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/MIROC-ES2H/baseline/r02
Processing scenario=baseline, member=r03


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/MIROC-ES2H/baseline/r03
